# IRP Stage 3 — Experiment Tracking
## Scenario 1: A single data scientist participating in an ML competition
**MLOps Group Project | Section 1, Group 5**  
*Maria-Irina Popa, Enzo Jerez, Roberto Cummings, Jia Yi Rachel Lee, Thomas Christian Matenco*

---

need to run this in terminal
mlflow server \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlartifacts \
  --host 127.0.0.1 \
  --port 5001

In [ ]:
import mlflow
import mlflow.sklearn
import sklearn
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("inventory-risk-stage3-team")

/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location=('/Users/ljyr/Documents/IE/Term 2/1 MLOPS Machine Learning '
 'Operations/IRP-Section1Group5/03-experiment-tracking/mlruns/1'), creation_time=1774312385929, experiment_id='1', last_update_time=1774312385929, lifecycle_stage='active', name='inventory-risk-stage3', tags={}, workspace='default'>

In [2]:
train = pd.read_parquet("../02-features-modelling/data/train.parquet")
val   = pd.read_parquet("../02-features-modelling/data/val.parquet")
test  = pd.read_parquet("../02-features-modelling/data/test.parquet")

In [3]:
print(train.shape, val.shape, test.shape)
print(train.head(2))

(53900, 30) (12300, 30) (6100, 30)
        Date Store ID Product ID     Category Region  Inventory Level  \
0 2022-01-08     S001      P0001    Furniture   East              231   
1 2022-01-09     S001      P0001  Electronics  South              373   

   Units Sold  Units Ordered  Demand Forecast  Price  ...  Inventory_Change  \
0           2            119             0.84  66.30  ...              -1.0   
1         350            178           352.24  41.72  ...             117.0   

  Inventory_Change_Pct  Days_of_Stock  Inventory_vs_Rolling7 Sales_Velocity  \
0            -0.001965     254.000000             173.857143       0.003937   
1             0.230315       1.785714             251.285714       0.560000   

   Coverage_Ratio  Forecast_Error  Order_to_Inventory  Risk_Label_Current  \
0      508.000000            1.16            0.234252           Safe Zone   
1        1.774358           -2.24            0.284800       Stockout Risk   

      Risk_Label  
0  Stockout Risk  

In [4]:
TARGET = "Risk_Label"

drop_cols = [
    "Risk_Label",
    "Risk_Label_Current",
    "Store ID",
    "Product ID",
    "Date",
    "Demand Forecast",
    "Demand_Forecast_Clean"
]

numerical_features = [
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Price",
    "Discount",
    "Units_Sold_Lag1",
    "Inventory_Change_Pct",
    "Days_of_Stock",
    "Sales_Velocity",
    "Coverage_Ratio",
    "Forecast_Error",
    "Order_to_Inventory"
]

categorical_features = [
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality"
]

final_features = numerical_features + categorical_features

X_train = train.drop(columns=drop_cols)[final_features].copy()
X_val   = val.drop(columns=drop_cols)[final_features].copy()
X_test  = test.drop(columns=drop_cols)[final_features].copy()

y_train_raw = train[TARGET].copy()
y_val_raw   = val[TARGET].copy()
y_test_raw  = test[TARGET].copy()

print(X_train.shape, X_val.shape, X_test.shape)
print(X_train.columns.tolist())

(53900, 16) (12300, 16) (6100, 16)
['Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Units_Sold_Lag1', 'Inventory_Change_Pct', 'Days_of_Stock', 'Sales_Velocity', 'Coverage_Ratio', 'Forecast_Error', 'Order_to_Inventory', 'Category', 'Region', 'Weather Condition', 'Seasonality']


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)

class_names = list(le.classes_)
print(class_names)

['Overstock Risk', 'Safe Zone', 'Stockout Risk']


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [7]:
numeric_preprocessor_logit = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_preprocessor_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_logit = ColumnTransformer([
    ("num", numeric_preprocessor_logit, numerical_features),
    ("cat", categorical_preprocessor, categorical_features)
])

preprocessor_tree = ColumnTransformer([
    ("num", numeric_preprocessor_tree, numerical_features),
    ("cat", categorical_preprocessor, categorical_features)
])

In [8]:
logit_pipeline = Pipeline([
    ("preprocessor", preprocessor_logit),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(class_names),
        random_state=42,
        eval_metric="mlogloss"
    ))
])

In [9]:
models = [
    ("Logistic Regression", logit_pipeline),
    ("Random Forest", rf_pipeline),
    ("XGBoost", xgb_pipeline)
]

In [10]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report

results = []

for model_name, model in models:
    with mlflow.start_run(run_name=model_name) as run:
        mlflow.set_tag("model_type", model_name)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        mlflow.log_param("numerical_features", ", ".join(numerical_features))
        mlflow.log_param("categorical_features", ", ".join(categorical_features))
        mlflow.log_param("n_classes", len(np.unique(y_train)))

        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        train_acc = accuracy_score(y_train, y_train_pred)
        val_acc = accuracy_score(y_val, y_val_pred)
        val_f1 = f1_score(y_val, y_val_pred, average="macro")

        mlflow.log_metric("train_accuracy", train_acc)
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_f1_macro", val_f1)

        cm = confusion_matrix(y_val, y_val_pred)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        cm_df.to_csv("confusion_matrix_labeled.csv")
        mlflow.log_artifact("confusion_matrix_labeled.csv")

        plt.figure(figsize=(6, 5))
        sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Validation Confusion Matrix ({model_name})")
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig("confusion_matrix_heatmap.png")
        plt.close()
        mlflow.log_artifact("confusion_matrix_heatmap.png")

        report = classification_report(y_val, y_val_pred, target_names=class_names, zero_division=0)
        with open("classification_report.txt", "w") as f:
            f.write(report)
        mlflow.log_artifact("classification_report.txt")

        input_example = X_train.iloc[[0]]
        mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example)

        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("description", f"{model_name} on inventory risk data")

        print(f"Logged run for {model_name}, val_accuracy={val_acc:.3f}, val_f1_macro={val_f1:.3f}")

        results.append({
            "model_name": model_name,
            "run_id": run.info.run_id,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "val_f1_macro": val_f1
        })

2026/03/24 01:41:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 01:41:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternat

Logged run for Logistic Regression, val_accuracy=0.273, val_f1_macro=0.244


2026/03/24 01:41:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 01:41:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternat

Logged run for Random Forest, val_accuracy=0.523, val_f1_macro=0.328


2026/03/24 01:41:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 01:41:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternat

Logged run for XGBoost, val_accuracy=0.739, val_f1_macro=0.283


In [11]:
results_df = pd.DataFrame(results).sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
results_df

,model_name,run_id,train_accuracy,val_accuracy,val_f1_macro
0,Random Forest,042319c7d6304a309235f16d22d82a13,0.758460,0.523008,0.328328
1,XGBoost,8384a8d5d32e4dc8ae3186984eb4bcdc,0.736531,0.739268,0.283364
2,Logistic Regression,7c47a8e4414a46ef91c9bffe4af95842,0.276790,0.273415,0.243865


In [12]:
best_run = results_df.iloc[0]
best_model_name = best_run["model_name"]
best_run_id = best_run["run_id"]

print("Best model:", best_model_name)
print("Best run_id:", best_run_id)

with open("run_id.txt", "w") as f:
    f.write(best_run_id)

Best model: Random Forest
Best run_id: 042319c7d6304a309235f16d22d82a13
